# KNeighborsClassifier am Beispiel des Iris Datensatzes


In diesem Jupyter Notebook wird der **K-Nearest-Neighbor-Algorithmus** (**KNN**, zu Deutsch „k-nächste-Nachbarn-Algorithmus“, siehe [Wikipedia - deutsch](https://de.wikipedia.org/wiki/N%C3%A4chste-Nachbarn-Klassifikation) und [Wikipedia - englisch](https://en.wikipedia.org/wiki/K-nearest_neighbors_algorithm)) 
am Beispiel des _Schwertlilien_ Datensatzes 
(siehe [Wikipedia - Iris flower data set](https://en.wikipedia.org/wiki/Iris_flower_data_set) ) und 
in der scikit-learn Implementierung
(siehe [Scikit-learn - KNeighborsClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html))
vorgestellt.

Der  **K-Nearest-Neighbor-Algorithmus** ist ein Klassifikationsverfahren, bei dem eine Klassenzuordnung unter Berücksichtigung seiner $k$ nächsten Nachbarn vorgenommen wird.
Die Anzahl $k$ der nächsten Nachbar ist ein Einstellparameter ("Hyperparameter") diese Klassifikationsverfahrens.

Das Lernen erfolgt durch einfaches Abspeichern der Trainingsbeispiele.

-----
2026-06-16 - ug V1.5


----
Referenzen
- [Nächste-Nachbarn-Klassifikation - Wikipedia](https://de.wikipedia.org/wiki/N%C3%A4chste-Nachbarn-Klassifikation) 
  [K-nearest_neighbors_algorithm - Wikipedia](https://en.wikipedia.org/wiki/K-nearest_neighbors_algorithm)) 
- [Scikit-learn - Wikipedia](https://de.wikipedia.org/wiki/Scikit-learn)
- [KNeighborsClassifier - Scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html)

<!--
Unterdrücken von Warnungen, die durch Seaborn 0.12.2 entstehen, siehe [seaborn - github - issue 3462](https://github.com/mwaskom/seaborn/issues/3462) und [Seaborn futurewarning caused by pandas dataframe](https://stackoverflow.com/questions/77882407/seaborn-futurewarning-caused-by-pandas-dataframe)
import warnings
warnings.filterwarnings("ignore", "is_categorical_dtype")
warnings.filterwarnings("ignore", "use_inf_as_na")
warnings.filterwarnings("ignore", category=FutureWarning, module="seaborn")
-->

## Vorbereitung
### Python Pakete laden

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

Anzeigen der verwendeten Seaborn Version:

In [ ]:
sns.__version__

### Datensatz laden

Der _Schwertlilien_ Datensatzes (siehe [Wikipedia - Iris flower data set](https://en.wikipedia.org/wiki/Iris_flower_data_set)) ist als Beispiel-Datensatz im [seaborn](https://seaborn.pydata.org/index.html)-Paket für statistische Datenvisualisierung enthalten und kann mit der Funktion [`sns.load_dataset()`](https://seaborn.pydata.org/generated/seaborn.load_dataset.html) als _Pandas-DataFrame_ geladen werden.

Der _Schwertlilien_ Datensatz enthält 5 Spalten:
- die Bezeichung der Schwertlinien-Spezies (_'species'_) und die
- Längen und Breiten jeweils der Blütenblätter (_'sepal_length'_, _'sepal_width'_, _'petal_length'_, _'petal_width'_)

Anmerkung: Blütenblätter unterteilen sich in 
- sepal [Kelchblatt](https://de.wikipedia.org/wiki/Kelchblatt)
- pedal [Kronblatt](https://de.wikipedia.org/wiki/Kronblatt)

In [ ]:
iris = sns.load_dataset('iris')

In [ ]:
iris.head()

In [ ]:
type(iris)

Welche unterschiedlichen Schwertlinien-Spezies haben wir im Datensatz?

In [ ]:
iris['species'].unique()

## Machine Learning - Aufgabenstellung

Wir wollen im Folgenden einen **Klassifikator** (ein Modell) entwickeln, mit dem sich die Spezies der drei Schwertlilie 
(_'setosa'_, _'versicolor'_ und _'virginica'_) 
aus den Daten (Länge und Breite) der Blütenblätter (Kelch- und Kronblatt) voraussagen läßt.

### Pairplot

Wir schauen uns die Verteilung der Blütenblatt-Parameter der Spezies in einem Seaborn Pairplot [`sns.pairplot()`](https://seaborn.pydata.org/generated/seaborn.pairplot.html) an.
Durch den Parameter `hue='species'` werden die Spezies farblich unterschiedlich voneinander dargestellt.

_Hinweis zu transparente Darstellung der Punkte in den Scatterplots:_
Die transparente Darstellung der Punkte erfolgt durch den Parameter `alpha=0.3`, jedoch muss dieser in ein Python Dictionary geschrieben werden und dies Dictionary muss über den Parameter `plot_kws=` übergeben werden, siehe auch [How to adjust transparency (alpha) in seaborn pairplot? - stackoverflow](https://stackoverflow.com/questions/47200033/how-to-adjust-transparency-alpha-in-seaborn-pairplot).

In [ ]:
sns.pairplot(iris,hue='species',palette='rainbow', plot_kws={'alpha':0.3})

**Anmerkung**: Aus dem Pairplot ist gut zu erkennen, dass die Spezies _setosa_ gut von den anderen beiden separiertbar ist. Die Spezies _versicolor_ und _virginica_ überlappen sich in einem Bereich, d.h. die 4 Features sind hier für eine klare Unterscheidung nicht ausreichend. Dann kann auch der Klassifikator in diesen Bereichen keine zuverlässigen Voraussagen machen.

####  Speichern des Pairplots für den Foliensatz:

In [ ]:
#plt.savefig('iris_pairplot.png', facecolor='white')

### Auswahl der Features und des Targets

Für den Klassifikator wählen wir folgende Features und folgendes Target: 
- als **Features $X$** wählen wird die 4 Spalten: _'sepal_length'_, _'sepal_width'_, _'petal_length'_, _'petal_width'_ und
- als **Target $y$** wählen wird die Spalte _'species'_.

Die Features können wir mit folgendem Befehl aus dem `iris`-DataFrame in einen neuen DataFrame extrahieren:

In [ ]:
X = iris[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']]
X.head()

Mit der [`.to_numpy()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_numpy.html)-Methode kann ein Pandas-DataFrame in ein _Numpy Array_ gewandelt werden:

In [ ]:
X = iris[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']].to_numpy()
X.shape

Die Targets `iris['species']` sind als Zeichenketten gespeichert: 

In [ ]:
iris['species'].head()

In [ ]:
iris['species'].unique()

Für die weitere Verabeitung mit Sklearn müssen die Zeichenketten in Zahlen gewandeln werden.

Mit der Funktion [`pd.Categorical()`](https://pandas.pydata.org/docs/reference/api/pandas.Categorical.html) läßt sich ein Pandas Series mit Zeichenketten in eine Pandas Series mit Kategorien wandeln:

In [ ]:
pd.Categorical(iris['species'])

Mit der [`.factorize()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.factorize.html)-Methode lassen sich die Kategorien in einer _Pandas Serie_ von Zeichenketten in Zahlen kodieren.

Die `.factorize()`-Methode hat zwei Rückgabe-Objekte:
- `y` - numerische Werte der Kategrie
- `mapping` - dies ist ein Categorical-Objekt und dient später die numerischen Wert wieder in Zeichenkette zu übersetzen

In [ ]:
y, mapping = pd.Categorical(iris['species']).factorize()
y

In [ ]:
type(mapping)

Mit dem Wert als Index erhält man die entsprechenden Zeichenkette:

In [ ]:
mapping[0], mapping[1], mapping[2]

Welche Dimension haben die Features `X` und das Target `y`:

In [ ]:
X.shape, y.shape

### Aufteilen der Daten in einen Trainings- und einen Testdatensatz

Mit der Funktion [`sklearn.model_selection.train_test_split`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) wird der Datensatz in einen Trainings- und einen Validierungs-Datensatz aufgeteilt.
Der Validierungs-Datensatz wird hier auch als Testdatensatz bezeichnet.

Mit dem Parameter `test_size=0.3` wird die relative Größe des Testdatensatzes zum gesamten Datensatz angegeben.
Mit dem Parameter `random_state=102` wird der Zufallsgenerator initialisiert. Dadurch, dass reproduzierbare Zufallswerte erzeugt werden, kommen bei den verschiedenen Durchläufen auch die gleiche Ergebnisse raus. Dies ist wichtig für die Fehlersuche bei der Entwicklung.

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=102)

In [ ]:
X_train.shape, y_train.shape

In [ ]:
X_test.shape, y_test.shape

### Auswahl des Klassifikations-Algorithmus

Wir wählen den KNeighborsClassifier aus dem Scikit-Learn Paket:
[`sklearn.neighbors.KNeighborsClassifier()`](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html)

In [ ]:
from sklearn import neighbors
knn = neighbors.KNeighborsClassifier()

Trainieren des Modells mit dem Trainingsdatensatz `X_train` und `y_train`:

In [ ]:
knn.fit(X_train, y_train)

### Evaluierung mit dem Testdatensatz

Aufruf der Funktion  [`confusion_matrix()`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html)  mit dem Array der _tatsächlichen_ Werte als erstem Parameter und dem Array der _vorausgesagten_ Werte als zweitem Parameter: 


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
cm = confusion_matrix(y_test, knn.predict(X_test))
cm

#### Plotten der Confusion Matrix mit `ConfusionMatrixDisplay`

Mit [`sklearn.metrics.ConfusionMatrixDisplay`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html) wird die Confusion Matrix als ein Heatmap-Plot dargestellt.

Instanz der [`sklearn.metrics.ConfusionMatrixDisplay`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html)-Klasse erstellen und mit der [.plot()](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html#sklearn.metrics.ConfusionMatrixDisplay.plot)-Methode den Plot der Confusion Matrix erzeugen.

In [ ]:
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=knn.classes_)
disp.plot()

#### Classification report
Mit der Funktion [`sklearn.metrics.classifikation_report()`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html)
werden die Kennzahlen berechnet und in Form eines Testberichtes ausgegeben:

Mit dem Parameter `target_names=` können die Namen der Klassen vorgegeben werden.

In [ ]:
print(classification_report(y_test, knn.predict(X_test), target_names=mapping))

#### Alternative Darstellung der Confusion-Matrix als Headmap

zuerst die Confusion-Matrix als _Pandas DataFrame_ erstellen:

In [ ]:
cm = confusion_matrix(y_test, knn.predict(X_test))
df_cm = pd.DataFrame(cm, index=mapping, columns=mapping)
df_cm

und dann als Headmap darstellen:

In [ ]:
sns.heatmap(df_cm, annot=True, fmt='d')

## Tuning der Hyperparameter

Der  **K-Nearest-Neighbor-Algorithmus** ist ein Klassifikationsverfahren, bei dem eine Klassenzuordnung unter Berücksichtigung seiner $k$ nächsten Nachbarn vorgenommen wird.

Die Anzahl $k$ der nächsten Nachbar ist ein Einstellparameter ("Hyperparameter") und kann durch den Parameter `n_neighbors=` beim Aufruf von [`sklearn.neighbors.KNeighborsClassifier()`](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html) vorgegeben werden.
Defaultmäßig wird `n_neighbors` auf 5 gesetzt.




In [ ]:
knn2 = neighbors.KNeighborsClassifier(n_neighbors=4)
knn2.fit(X_train, y_train)
print(classification_report(y_test, knn2.predict(X_test), target_names=mapping))
cm2 = confusion_matrix(y_test, knn2.predict(X_test))

disp = ConfusionMatrixDisplay(confusion_matrix=cm2, display_labels=knn2.classes_)
disp.plot()

#sns.heatmap(pd.DataFrame(cm, index=mapping, columns=mapping), annot=True, fmt='d');

Anmerkung: Durch Wahl von `n_neighbors=4` kann das Ergebnis leicht verbessert werden. Dies ist aber eher Glücksache, vergleiche dazu die Anmerkung zu dem eingangs erstellten Pairplot.

----
## Anwenden des erstellten Modells

#### Prädiktion machen

In [ ]:
print("Welche Art von Schwertlilie (Iris Setosa, Iris Virginica oder Iris Versicolor)?")
sepal_length = 3
sepal_width = 5
petal_length = 4
petal_width = 2

print(f"hat ein {sepal_length}cm x {sepal_width}cm Sepalum (Kelchblatt) und ein {petal_length}cm x {petal_width}cm Petalum (Kronblatt)")

X_neu = np.array([[sepal_length,sepal_width,petal_length,petal_width]])
y_pred = knn.predict(X_neu)
y_pred_string = mapping[y_pred[0]]
    
print(f"Antwort: {y_pred_string}")